# Myanmar Legal RAG — BGE-M3 GPU embedding

This notebook embeds any exported legal chunk file with the exact `bge-m3-q4` GGUF used by local Ollama and produces compatible 1,024-dimensional vectors for PostgreSQL. Before running, choose **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import csv
import hashlib
import json
import math
import os
import subprocess
import time
from pathlib import Path

import requests
from google.colab import files

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('GPU is not enabled. Select Runtime > Change runtime type > T4 GPU, then run again.')
print('GPU:', gpu.stdout.strip())

subprocess.run('apt-get update -qq && apt-get install -y -qq zstd', shell=True, check=True)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
ollama_env = {**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434'}
server = subprocess.Popen(['ollama', 'serve'], env=ollama_env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('Ollama server did not start')

gguf_url = 'https://huggingface.co/lm-kit/bge-m3-gguf/resolve/main/bge-m3-Q4_K_M.gguf'
gguf_path = Path('/content/bge-m3-Q4_K_M.gguf')
expected_sha256 = 'e251234fcb7d050991a6be491952f485bf5c641dd10c3272dc1301fd281ad50f'
with requests.get(gguf_url, stream=True, timeout=120) as response:
    response.raise_for_status()
    with gguf_path.open('wb') as handle:
        for block in response.iter_content(chunk_size=1024 * 1024):
            if block:
                handle.write(block)
actual_sha256 = hashlib.sha256(gguf_path.read_bytes()).hexdigest()
assert actual_sha256 == expected_sha256, actual_sha256
Path('/content/Modelfile.bge-m3-q4').write_text(f'FROM {gguf_path}\n', encoding='utf-8')
subprocess.run(['ollama', 'create', 'bge-m3-q4', '-f', '/content/Modelfile.bge-m3-q4'], env=ollama_env, check=True)

print('Exact Q4 model installed and checksum verified')

In [ ]:
uploaded = files.upload()
input_name = 'legal_chunks_for_colab.json'
if input_name not in uploaded:
    json_files = [name for name in uploaded if name.lower().endswith('.json')]
    if len(json_files) != 1:
        raise FileNotFoundError(f'Please upload {input_name}, or exactly one .json chunk export')
    input_name = json_files[0]

with open(input_name, 'r', encoding='utf-8') as handle:
    chunks = json.load(handle)

if not chunks:
    raise ValueError('Chunk export is empty')

required_fields = {'document_id', 'chunk_index', 'content'}
missing = [i for i, chunk in enumerate(chunks) if not required_fields.issubset(chunk)]
if missing:
    raise ValueError(f'Missing required fields in chunk rows: {missing[:10]}')

seen_keys = set()
by_document = {}
for chunk in chunks:
    key = (chunk['document_id'], int(chunk['chunk_index']))
    if key in seen_keys:
        raise ValueError(f'Duplicate document/chunk key: {key}')
    seen_keys.add(key)
    by_document.setdefault(chunk['document_id'], []).append(int(chunk['chunk_index']))

for document_id, indexes in by_document.items():
    expected = list(range(len(indexes)))
    if sorted(indexes) != expected:
        raise ValueError(f'Document {document_id} has non-contiguous chunk indexes')

texts = [chunk['content'] for chunk in chunks]
print(f'Loaded {len(texts)} chunks from {len(by_document)} document(s): {input_name}')

In [ ]:
started = time.time()
embeddings = []
batch_size = 16
for start in range(0, len(texts), batch_size):
    batch = texts[start:start + batch_size]
    response = requests.post(
        'http://127.0.0.1:11434/api/embed',
        json={
            'model': 'bge-m3-q4',
            'input': batch,
            'truncate': True,
            'keep_alive': '30m',
        },
        timeout=600,
    )
    response.raise_for_status()
    batch_vectors = response.json()['embeddings']
    if len(batch_vectors) != len(batch):
        raise RuntimeError(f'Batch {start} returned {len(batch_vectors)} vectors for {len(batch)} texts')
    embeddings.extend(batch_vectors)
    print(f'{len(embeddings)}/{len(texts)}')

assert len(embeddings) == len(chunks)
assert all(len(vector) == 1024 for vector in embeddings)
assert not any(math.isnan(float(value)) for vector in embeddings for value in vector)
print(f'Embedded {len(chunks)} chunks in {(time.time() - started) / 60:.1f} minutes')

In [ ]:
output_name = 'legal_chunks_embedded.csv'
fieldnames = [
    'document_id', 'chunk_index', 'chapter', 'section', 'subsection',
    'content', 'embedding', 'metadata'
]

with open(output_name, 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for chunk, vector in zip(chunks, embeddings):
        writer.writerow({
            'document_id': chunk['document_id'],
            'chunk_index': chunk['chunk_index'],
            'chapter': chunk.get('chapter') or '',
            'section': chunk.get('section') or '',
            'subsection': chunk.get('subsection') or '',
            'content': chunk['content'],
            'embedding': json.dumps(vector, separators=(',', ':')),
            'metadata': json.dumps(chunk.get('metadata', {}), ensure_ascii=False, separators=(',', ':')),
        })

print(f'Created {output_name}: {os.path.getsize(output_name) / 1024 / 1024:.1f} MB')
files.download(output_name)